# Raw Data Exploration

**Question**: What do we have? Before writing loading or cleaning code, we need to understand the raw files: their structure, formats, units, quirks, and potential problems.

**Why this notebook exists**: Every decision in `data_loader.py` should trace back to something discovered here. If we multiply by 1e6, it's because we found the units here. If we parse WKT, it's because we saw the format here. Nothing should be assumed.

**Rule**: This notebook uses only `pd.read_csv()`, `pd.read_excel()`, and basic Python. No `data_loader.py`, no `schemas.py`. We are looking at the data as delivered.

---

## Setup

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import ast
from pathlib import Path

# Deliberately NO imports from src/ — raw exploration only
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REFERENCE_DATA_DIR = PROJECT_ROOT / "data" / "reference"

RADNETT_RAW_PATH = next(RAW_DATA_DIR.glob("Radnett_AlleStasjoner_Overv*kingsdata_2023.csv"))
STATION_LOCATIONS_PATH = next(RAW_DATA_DIR.glob("Radnett*stasjoner*lokasjon.xlsx"))
CIVIL_DEFENCE_RAW_PATH = next(RAW_DATA_DIR.glob("Sivilforsvaret_M*lingsdata.csv"))
REPORT_STATION_PATH = REFERENCE_DATA_DIR / "dsa_report_station_positions.csv"

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 60)

---
## Part 1: Radnett Monitoring Data

### 1.1 First look
Always start by looking at the first few rows exactly as they appear.

In [10]:
# encoding="utf-8-sig" handles the BOM (byte order mark) Windows CSV exports add
radnett_raw = pd.read_csv(
    RADNETT_RAW_PATH,
    encoding="utf-8-sig"

)

print("First 5 rows:")
print(radnett_raw.head().to_string())
print(f"\nShape: {radnett_raw.shape}")

First 5 rows:
   Station Code          Station Name              Time  Dose rate [microSv/h]
0             3  Østerås (Luftfilter)  01.01.2023 00:00                    0.0
1             3  Østerås (Luftfilter)  01.01.2023 01:00                    0.0
2             3  Østerås (Luftfilter)  01.01.2023 02:00                    0.0
3             3  Østerås (Luftfilter)  01.01.2023 03:00                    0.0
4             3  Østerås (Luftfilter)  01.01.2023 04:00                    0.0

Shape: (385484, 4)


**Observations**:
- 4 columns: Station Code, Station Name, Time, Dose rate
- Time format is `DD.MM.YYYY HH:MM` (European, not ISO)
- Dose rate is numeric

### 1.2 Column types and basic statistics

In [ ]:
print("Column types as pandas read them:")
print(radnett_raw.dtypes)
print(f"\nTotal rows: {radnett_raw.shape[0]:,}")
print(f"\nNull values per column:")
print(radnett_raw.isna().sum())
print(f"\nDose rate statistics:")
print(radnett_raw["Dose rate [microSv/h]"].describe())

**Note**: `Time` is read as string — needs parsing. No null values found by pandas — but missing data may be coded as 0.

### 1.3 Station inventory

In [ ]:
stations = radnett_raw["Station Name"].value_counts()
print(f"Unique stations: {len(stations)}")
print(f"\nReadings per station:")
print(stations.to_string())
print(f"\nAll same count? {stations.nunique() == 1}")
print(f"8761 hours = 365 days x 24h + 1 = complete structural year")

**Key finding**: Every station has exactly 8,761 rows. The data is *structurally complete* — every hour has a row. Missing data is hidden inside the values, not as missing rows.

### 1.4 Station types from naming convention

In [ ]:
print("All station names:")
for name in sorted(radnett_raw["Station Name"].unique()):
    if "(Luftfilter)" in name:
        tag = "<-- AIR FILTER"
    elif "Mobil" in name:
        tag = "<-- MOBILE"
    else:
        tag = ""
    print(f"  {name:35s} {tag}")

n_filter = sum("(Luftfilter)" in n for n in radnett_raw["Station Name"].unique())
n_mobile = sum("Mobil" in n for n in radnett_raw["Station Name"].unique())
n_fixed = radnett_raw["Station Name"].nunique() - n_filter - n_mobile
print(f"\nFixed: {n_fixed}, Air filter: {n_filter}, Mobile: {n_mobile}")

### 1.5 Investigating zero values — the critical question

Background radiation is always present (~0.05-0.15 uSv/h in Norway). A reading of exactly 0.000 means the detector was not reporting.

In [ ]:
total = radnett_raw.groupby("Station Name").size()
zeros = radnett_raw[radnett_raw["Dose rate [microSv/h]"] == 0].groupby("Station Name").size()

print("Zero values per station (sorted by severity):")
print(f"  {'Station':35s} {'Zeros':>7s} / {'Total':>5s}  {'Pct':>7s}  Status")
print("  " + "-" * 75)

for station in zeros.sort_values(ascending=False).index:
    z = zeros[station]
    t = total[station]
    pct = z / t * 100
    if pct > 50:
        status = "OFFLINE"
    elif pct > 5:
        status = "UNSTABLE"
    else:
        status = ""
    print(f"  {station:35s} {z:5d} / {t:5d}  {pct:6.1f}%  {status}")

n_offline = sum(1 for s in zeros.index if zeros[s] / total[s] > 0.5)
n_unstable = sum(1 for s in zeros.index if 0.05 < zeros[s] / total[s] <= 0.5)
print(f"\nOffline (>50% zeros): {n_offline} stations")
print(f"Unstable (5-50% zeros): {n_unstable} stations")

**This is the most important finding in the entire exploration.**

Zero values = missing data, not zero radiation. Some stations were completely offline.

### 1.6 Visual comparison: healthy vs failing station

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

oslo = radnett_raw[radnett_raw["Station Name"] == "Oslo"].copy()
oslo["t"] = pd.to_datetime(oslo["Time"], format="%d.%m.%Y %H:%M")
axes[0].plot(oslo["t"], oslo["Dose rate [microSv/h]"], lw=0.3, color="#2196F3")
axes[0].set_ylabel("Dose rate (uSv/h)")
axes[0].set_title("Oslo - reliable station")

kjeller = radnett_raw[radnett_raw["Station Name"] == "Kjeller"].copy()
kjeller["t"] = pd.to_datetime(kjeller["Time"], format="%d.%m.%Y %H:%M")
axes[1].plot(kjeller["t"], kjeller["Dose rate [microSv/h]"], lw=0.3, color="#F44336")
axes[1].set_ylabel("Dose rate (uSv/h)")
axes[1].set_title("Kjeller - mostly zeros (offline)")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

print("Oslo shows seasonal pattern (lower in winter = snow shielding) and spikes (radon washout)")
print("Kjeller is flat at zero with brief data periods - confirms offline, not zero radiation")

### 1.7 Time format verification

In [ ]:
sample_time = radnett_raw["Time"].iloc[0]
parsed = pd.to_datetime(sample_time, format="%d.%m.%Y %H:%M")
print(f"Raw: {sample_time!r}")
print(f"Parsed: {parsed}")
print(f"Format string for pandas: %d.%m.%Y %H:%M")

### 1.8 Non-zero dose rate distribution

In [ ]:
nonzero = radnett_raw[radnett_raw["Dose rate [microSv/h]"] > 0]["Dose rate [microSv/h]"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(nonzero, bins=100, edgecolor="white", alpha=0.7)
axes[0].set_xlabel("Dose rate (uSv/h)")
axes[0].set_title("Distribution of non-zero readings")
axes[0].axvline(nonzero.median(), color="red", ls="--", label=f"Median: {nonzero.median():.3f}")
axes[0].legend()

means = radnett_raw[radnett_raw["Dose rate [microSv/h]"] > 0].groupby("Station Name")["Dose rate [microSv/h]"].mean().sort_values()
axes[1].barh(range(len(means)), means.values, color="#2196F3")
axes[1].set_yticks(range(len(means)))
axes[1].set_yticklabels(means.index, fontsize=7)
axes[1].set_xlabel("Mean dose rate (uSv/h)")
axes[1].set_title("Per-station mean (non-zero)")

plt.tight_layout()
plt.show()
print(f"Range: {nonzero.min():.4f} to {nonzero.max():.4f} uSv/h - all within normal background")

---
## Part 2: Station Locations

### 2.1 Raw file

In [5]:
loc_raw = pd.read_excel(STATION_LOCATIONS_PATH, engine="openpyxl")
print(loc_raw.to_string())
print(f"\nColumn types:")
print(loc_raw.dtypes)

                  Stasjon    VertPos    HorzPos
0    Østerås (Luftfilter)  59.947940  10.602880
1       Sola (Luftfilter)  58.880841   5.645943
2   Svanhovd (Luftfilter)  69.455132  30.040915
3                 Svolvær  68.232198  14.562573
4    Skibotn (Luftfilter)  69.371735  20.303785
5                   Runde  62.398990   5.660660
6                 Arendal  58.476399   8.817814
7                    Oslo  59.943100  10.721150
8                Svanhovd  69.455161  30.041229
9              Hammerfest  70.671266  23.666003
10                 Tromsø  69.653669  18.936566
11               Karasjok  69.463515  25.502517
12           Longyearbyen  78.222626  15.624433
13                Kjeller  59.976139  11.050119
14                  Hitra  63.638532   8.687296
15             Kautokeino  69.021938  23.019255
16            Haakonsvern  60.335540   5.234538
17              Sørkjosen  69.785608  20.941505
18    Mobil målestasjon 1  59.947950  10.602830
19    Mobil målestasjon 2  60.000000  10

### 2.2 Coordinate type check

In [ ]:
print("Checking for non-float coordinates:")
found = False
for _, row in loc_raw.iterrows():
    lt = type(row["VertPos"]).__name__
    lo = type(row["HorzPos"]).__name__
    if lt != "float" or lo != "float":
        found = True
        print(f"  {row['Stasjon']:35s} lat={row['VertPos']!r} ({lt}), lon={row['HorzPos']!r} ({lo})")
if not found:
    print("  All coordinates are float")
else:
    print("\n--> Pipeline must use pd.to_numeric() to force float types")

### 2.3 Duplicate/placeholder coordinates

In [ ]:
groups = loc_raw.groupby(["VertPos", "HorzPos"])["Stasjon"].apply(list)
dupes = groups[groups.apply(len) > 1]
if len(dupes) > 0:
    print("Shared coordinates:")
    for (lat, lon), names in dupes.items():
        print(f"  ({lat}, {lon}): {names}")
    print("\n--> Placeholder coords for mobile stations. Must flag in pipeline.")
else:
    print("No duplicates found")

### 2.4 Station name matching

In [ ]:
r_names = set(radnett_raw["Station Name"].unique())
l_names = set(loc_raw["Stasjon"].unique())
diff1 = r_names - l_names
diff2 = l_names - r_names
if not diff1 and not diff2:
    print(f"All {len(r_names)} station names match perfectly.")
else:
    print(f"In Radnett only: {diff1}")
    print(f"In locations only: {diff2}")

---
## Part 3: Civil Defence Data

Most complex dataset — several parsing challenges.

### 3.1 First look

In [6]:
civil_raw = pd.read_csv(CIVIL_DEFENCE_RAW_PATH)
print("First 3 rows (transposed):")
print(civil_raw.head(3).T.to_string())
print(f"\nShape: {civil_raw.shape}")
print(f"\nColumn types:")
print(civil_raw.dtypes)

First 3 rows (transposed):
                                                                                                                                                                                                                                                                                                                                                        0                                                                                                                                                                                                                                                                                                                                             1                                                                                                                                                                                                                                                                                                      

### 3.2 Location column — parsing WKT coordinates

Format: `SRID=4326;POINT(longitude latitude)`. **Longitude first!** (WKT standard)

In [ ]:
print("Sample locations:")
for i in range(3):
    print(f"  {civil_raw['location'].iloc[i]}")

# Manual parse of first entry
sample = civil_raw["location"].iloc[0]
match = re.search(r"POINT\(([-\d.]+)\s+([-\d.]+)\)", sample)
lon, lat = float(match.group(1)), float(match.group(2))
print(f"\nParsed: lon={lon:.4f}, lat={lat:.4f}")
print(f"IMPORTANT: Inside POINT(), longitude comes FIRST, latitude SECOND")

### 3.3 Parse all coordinates and find problems

In [7]:
def parse_point(s):
    try:
        m = re.search(r"POINT\(([-\d.]+)\s+([-\d.]+)\)", s)
        if m:
            return float(m.group(2)), float(m.group(1))
    except Exception:
        pass
    return np.nan, np.nan

coords = civil_raw["location"].apply(parse_point)
civil_raw["lat"] = coords.apply(lambda x: x[0])
civil_raw["lon"] = coords.apply(lambda x: x[1])

print(f"Lat range: {civil_raw['lat'].min():.4f} to {civil_raw['lat'].max():.4f}")
print(f"Lon range: {civil_raw['lon'].min():.4f} to {civil_raw['lon'].max():.4f}")

# Flag problems
suspect = civil_raw[
    (civil_raw["lat"] < 55) | (civil_raw["lat"] > 82) |
    (civil_raw["lon"] < 3) | (civil_raw["lon"] > 35)
]
print(f"\nCoordinates outside Norway/Svalbard: {len(suspect)}")
for _, row in suspect.iterrows():
    try:
        meta = ast.literal_eval(row["metadata"])
        name = meta.get("measuring_point_name", "?")
    except Exception:
        name = "?"
    issue = "GPS fail (0,0)" if row["lat"] == 0 else "Coord error"
    print(f"  ({row['lat']:8.4f}, {row['lon']:8.4f}) | {name:40s} | {issue}")

Lat range: 0.0000 to 71.0642
Lon range: 0.0000 to 57.2413

Coordinates outside Norway/Svalbard: 13
  (  0.0000,   0.0000) | Klungset, Fauske                         | GPS fail (0,0)
  ( 60.2653,   2.7898) | Uvdal, Nore og Uvdal                     | Coord error
  (  0.0000,   0.0000) | Horn Brønnøysund                         | GPS fail (0,0)
  (  1.8306,  57.1619) | Leirvik, Stord                           | Coord error
  (  1.7761,  57.2413) | Rimbareid, Fitjar                        | Coord error
  (  0.0000,   0.0000) | Båsmo barnehage parkering nord           | GPS fail (0,0)
  ( 60.2653,   2.7898) | Uvdal, Nore og Uvdal                     | Coord error
  (  0.0000,   0.0000) | Båsmo barnehage parkering nord           | GPS fail (0,0)
  (  0.0000,   0.0000) | Horn Brønnøysund                         | GPS fail (0,0)
  (  0.0000,   0.0000) | Båsmo barnehage parkering nord           | GPS fail (0,0)
  (  0.0000,   0.0000) | Båsmo barnehage parkering nord           | GPS fail (0,0)


### 3.4 Dose rate units — the critical trap

Radnett uses **µSv/h**. Civil Defence uses **Sv/h**. Factor 1,000,000 difference.

In [ ]:
print("Raw Civil Defence dose rate (Sv/h):")
print(civil_raw["doserate [Sv/h]"].describe())

civil_raw["dose_microsv"] = civil_raw["doserate [Sv/h]"] * 1e6
print(f"\nConverted to uSv/h:")
print(civil_raw["dose_microsv"].describe())

r_median = radnett_raw[radnett_raw["Dose rate [microSv/h]"] > 0]["Dose rate [microSv/h]"].median()
c_median = civil_raw["dose_microsv"].median()
print(f"\nSanity check:")
print(f"  Radnett median:       {r_median:.4f} uSv/h")
print(f"  Civil Defence median: {c_median:.4f} uSv/h")
print(f"  Same order of magnitude: {0.3 < r_median/c_median < 3.0}")

### 3.5 Metadata column — what is hidden in there?

In [ ]:
# Parse one entry to see the structure
sample = ast.literal_eval(civil_raw["metadata"].iloc[0])
print("Metadata keys and types:")
for k, v in sample.items():
    print(f"  {k:30s} = {v!r:40s} ({type(v).__name__})")

# Check if ALL entries parse
ok, fail = 0, 0
rain_counts = {}
for s in civil_raw["metadata"]:
    try:
        p = ast.literal_eval(s)
        ok += 1
        r = p.get("rainfall")
        rain_counts[r] = rain_counts.get(r, 0) + 1
    except:
        fail += 1

print(f"\nParsing: {ok} success, {fail} failures")
print(f"Rainfall flag: {rain_counts}")
print(f"\n--> {rain_counts.get(True, 0)} measurements during rain ")
print(f"--> Analytically valuable for radon washout analysis")

### 3.6 Time range

In [ ]:
civil_raw["time_parsed"] = pd.to_datetime(civil_raw["timestamp"])
print(f"Range: {civil_raw['time_parsed'].min()} to {civil_raw['time_parsed'].max()}")
print(f"\nPer year:")
print(civil_raw["time_parsed"].dt.year.value_counts().sort_index())

### 3.7 Extreme values

In [ ]:
print("Top 10 highest readings (uSv/h):")
top = civil_raw.nlargest(10, "dose_microsv")
for _, row in top.iterrows():
    try:
        m = ast.literal_eval(row["metadata"])
        name = m.get("measuring_point_name", "NO NAME")
        rain = m.get("rainfall", "?")
    except:
        name, rain = "?", "?"
    print(f"  {row['dose_microsv']:.4f} | {str(row['timestamp'])[:10]} | rain={str(rain):5s} | {name}")

print(f"\nHighest is {civil_raw['dose_microsv'].max():.0f}x the median - needs investigation")

### 3.8 Other columns

In [ ]:
print("measurement_height:", civil_raw["measurement_height"].value_counts().to_dict())
print("measurement_type:", civil_raw["measurement_type"].value_counts().to_dict())
print("event:", civil_raw["event"].value_counts().to_dict())
print(f"Unique teams: {civil_raw['team'].nunique()}")
print("\n--> All FOT (on foot), all Sivilforsvaret, height 1.0 or 1.5m")

---
## Part 3B: Cross-checking station locations against DSA Report

The DSA annual report (2024:04, Table 1) lists coordinates for all 33 fixed Radnett stations in degrees and minutes format. We can cross-check the location file against this independent source.

This also gives us the **placement type** (ground vs building) which determines whether a station shows seasonal variation — information not available in the data file.

Reference data extracted from DSA Report 2024:04, Table 1, saved as `data/reference/dsa_report_station_positions.csv`.

In [8]:
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance in km between two lat/lon points."""
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# Load report reference (degrees + minutes from DSA Report 2024:04, Table 1)
report = pd.read_csv(REPORT_STATION_PATH)
report["report_lat"] = report["lat_deg"] + report["lat_min"] / 60
report["report_lon"] = report["lon_deg"] + report["lon_min"] / 60

# Load file locations
loc = pd.read_excel(STATION_LOCATIONS_PATH, engine="openpyxl")
loc.columns = ["station_name", "latitude", "longitude"]
loc["latitude"] = pd.to_numeric(loc["latitude"], errors="coerce")
loc["longitude"] = pd.to_numeric(loc["longitude"], errors="coerce")

# Match and compare
results = []
for _, r_row in report.iterrows():
    name = r_row["station_name"]
    match = loc[loc["station_name"] == name]
    if len(match) == 0:
        continue
    f_lat = match.iloc[0]["latitude"]
    f_lon = match.iloc[0]["longitude"]
    dist = haversine_km(f_lat, f_lon, r_row["report_lat"], r_row["report_lon"])
    results.append({
        "station": name,
        "file_lat": f_lat,
        "file_lon": f_lon,
        "report_lat": r_row["report_lat"],
        "report_lon": r_row["report_lon"],
        "distance_km": dist,
        "placement": r_row["placement"],
    })

results_df = pd.DataFrame(results).sort_values("distance_km", ascending=False)
print("Cross-check: Location file vs DSA Report 2024:04, Table 1")
print("Sorted by distance (largest discrepancy first):\n")
for _, row in results_df.iterrows():
    flag = ""
    if row["distance_km"] > 10:
        flag = "*** MISMATCH ***"
    elif row["distance_km"] > 5:
        flag = "* check *"
    print(
        f"  {row['station']:<25s} file=({row['file_lat']:7.3f},{row['file_lon']:7.3f})  "
        f"report=({row['report_lat']:7.3f},{row['report_lon']:7.3f})  "
        f"{row['distance_km']:6.1f}km  {row['placement']:>8s}  {flag}"
    )

Cross-check: Location file vs DSA Report 2024:04, Table 1
Sorted by distance (largest discrepancy first):

  Kautokeino                file=( 69.022, 23.019)  report=( 69.583, 25.317)   109.8km    ground  *** MISMATCH ***
  Sørkjosen                 file=( 69.786, 20.942)  report=( 69.583, 20.967)    22.5km  building  *** MISMATCH ***
  Arendal                   file=( 58.476,  8.818)  report=( 58.517,  8.900)     6.5km    ground  * check *
  Snåsa                     file=( 64.259, 12.372)  report=( 64.233, 12.383)     2.9km    ground  
  Svolvær                   file=( 68.232, 14.563)  report=( 68.217, 14.583)     1.9km  building  
  Runde                     file=( 62.399,  5.661)  report=( 62.383,  5.650)     1.8km    ground  
  Vinje                     file=( 59.616,  7.855)  report=( 59.600,  7.850)     1.8km    ground  
  Hol                       file=( 60.580,  8.411)  report=( 60.567,  8.400)     1.6km    ground  
  Halden                    file=( 58.996, 11.528)  report=(

### Findings from cross-check

**Two major mismatches** (where the file is actually CORRECT and the report has errors):

1. **Kautokeino** (~110 km off): The report lists coordinates 69°35'N, 25°19'E, but Kautokeino town is at ~69.01°N, 23.04°E — which matches the file. The report's coordinates are actually near Karasjok. Note that the report gives the same latitude (69°35') for both Kautokeino and Sørkjosen, which is suspicious.

2. **Sørkjosen** (~22 km off): Similar issue — the report's latitude appears incorrect. The file's position matches Sørkjosen's actual location.

3. **Arendal** (~6.5 km): Minor discrepancy, likely the station is outside town center.

**Bonus discovery**: The report tells us which stations are `ground` (bakkennivå) vs `building` (bygning). This is critical for our analysis — building-mounted stations don't show seasonal snow-shielding effects. Stations on buildings:
Hammerfest, Sørkjosen, Svolvær, Bodø, Mo i Rana, Bergen, Halden.

**For the pipeline**: We should add the `placement` column from the report to our station data. This lets us properly classify ground vs building stations instead of guessing.

---
## Part 4: Summary of Discoveries

### What drives the pipeline design

| # | Discovery | Where | Pipeline action |
|---|-----------|-------|-----------------|
| 1 | Zeros = missing data, not zero radiation | 1.5 | Handle per station |
| 2 | Time format DD.MM.YYYY HH:MM | 1.7 | Parse with `%d.%m.%Y %H:%M` |
| 3 | Station types from naming convention | 1.4 | Add station_type column |
| 4 | Stavanger coords stored as strings | 2.2 | Force pd.to_numeric() |
| 5 | Mobile stations 2-6 have placeholder (60,10) | 2.3 | Flag as placeholder |
| 6 | WKT POINT has lon first, lat second | 3.2 | Parse correctly |
| 7 | Civil Defence units Sv/h vs Radnett µSv/h | 3.4 | Multiply by 1e6 |
| 8 | Metadata is parseable dict with rainfall flag | 3.5 | Extract with ast.literal_eval |
| 9 | GPS failures give (0,0) coordinates | 3.3 | Flag invalid |
| 10 | Some coords swapped or corrupt | 3.3 | Flag invalid |
| 11 | Extreme outlier near Bodø (60x median) | 3.7 | Flag for investigation |
| 12 | DSA report has wrong coords for Kautokeino, Sørkjosen | 3B | File is correct, note discrepancy |
| 13 | Report gives ground/building placement per station | 3B | Add placement column to station data |

### Now update the Copilot instruction files

With these confirmed discoveries, we can add **structural data facts** (not analytical findings) to `.github/copilot-instructions.md`:
column names, types, unit difference, coordinate format, metadata structure, station type classification, and ground/building placement from the DSA report.